# GeoClean detector

Trains YOLOv8n on `data/dataset` (1,245 images, two classes: `roadside_dump` and `overflowing_bin`), built by `scripts/export_yolo_dataset.py`. Splits are group-disjoint: 907 train, 169 val and 169 test images.

**All labels are unreviewed** (`auto-v1`: Roboflow boxes plus bootstrap-detector boxes accepted without checking), including in val and test. Validation and test metrics therefore measure agreement with those labels, not accuracy on Pune scenes. The set has no hard negatives.

**Before running:** *Runtime -> Change runtime type -> T4 GPU*. Then *Runtime -> Run all*, and upload `data/zips/geoclean_dataset.zip` (about 162 MB) from your computer when the picker opens. If labels changed in Label Studio, rebuild it first with `scripts/export_yolo_dataset.py`.

In [ ]:
!pip install -q ultralytics

## 1. Upload and extract the dataset
Select `geoclean_dataset.zip` when the picker opens.

In [ ]:
from google.colab import files
import zipfile, os, shutil

def extract(zip_name, dest):
    # Normalise separators in case a Windows tool stored paths as images\train\x.jpg.
    shutil.rmtree(dest, ignore_errors=True)
    with zipfile.ZipFile(zip_name) as z:
        for info in z.infolist():
            name = info.filename.replace('\\', '/')
            if name.endswith('/'):
                continue
            target = os.path.join(dest, name)
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as out:
                shutil.copyfileobj(src, out)

uploaded = files.upload()
zip_name = next(iter(uploaded))
extract(zip_name, '/content/dataset')
for split in ('train', 'val', 'test'):
    folder = f'/content/dataset/images/{split}'
    if not os.path.isdir(folder):
        raise SystemExit(f'{folder} is missing. Did you upload geoclean_dataset.zip? You uploaded {zip_name}.')
    print(split, len(os.listdir(folder)), 'images')

## 2. Train
100 epochs at 640 px, as in the blueprint. Takes roughly 35-45 minutes on a T4. `patience=20` stops early if validation stops improving.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    seed=20260913,
    project='/content/runs',
    name='geoclean',
)

## 3. Score the best weights on val and test
Per-class precision, recall, mAP50 and mAP50-95, saved to `metrics.json`. Remember these are scored against unreviewed labels.

In [ ]:
import glob, json

# Ultralytics saves to geoclean2, geoclean3... when an earlier run folder exists, so use the newest best.pt.
candidates = glob.glob('/content/runs/**/weights/best.pt', recursive=True)
if not candidates:
    raise SystemExit('No trained best.pt found under /content/runs. Did the training cell finish?')
BEST = max(candidates, key=os.path.getmtime)
RUN_DIR = os.path.dirname(os.path.dirname(BEST))
print('using', BEST)

best = YOLO(BEST)
CLASSES = ['roadside_dump', 'overflowing_bin']
report = {'label_origin': 'auto-v1 (unreviewed)', 'weights': 'best.pt'}
for split in ('val', 'test'):
    m = best.val(data='/content/dataset/data.yaml', split=split, project='/content/runs', name=f'eval_{split}')
    per_class = {}
    for i, name in enumerate(CLASSES):
        p, r, map50, map5095 = m.box.class_result(i)
        per_class[name] = {'precision': round(float(p), 4), 'recall': round(float(r), 4),
                           'mAP50': round(float(map50), 4), 'mAP50-95': round(float(map5095), 4)}
    report[split] = {'mAP50': round(float(m.box.map50), 4), 'mAP50-95': round(float(m.box.map), 4),
                     'precision': round(float(m.box.mp), 4), 'recall': round(float(m.box.mr), 4),
                     'per_class': per_class}

with open(f'{RUN_DIR}/metrics.json', 'w') as f:
    json.dump(report, f, indent=1)
print(json.dumps(report, indent=1))

## 4. Download the results
One zip with the best weights, `metrics.json`, `results.csv` and the training and evaluation plots. Save it as `data/geoclean_train_result.zip` in the project folder.

In [ ]:
with zipfile.ZipFile('/content/geoclean_train_result.zip', 'w') as z:
    z.write(BEST, 'geoclean_best.pt')
    for name in ('metrics.json', 'results.csv', 'args.yaml'):
        if os.path.exists(f'{RUN_DIR}/{name}'):
            z.write(f'{RUN_DIR}/{name}', name)
    for p in glob.glob(f'{RUN_DIR}/*.png'):
        z.write(p, os.path.basename(p))
    for split in ('val', 'test'):
        for p in glob.glob(f'/content/runs/eval_{split}*/*.png'):
            z.write(p, f'eval_{split}/' + os.path.basename(p))

files.download('/content/geoclean_train_result.zip')